---
title: "Лабораторна робота №1. Ухвалення рішень в умовах невизначеності. Частина 1"
description: "Моніторинг та керування в слабкоструктурованих процесах і системах | КрНУ ім. М. Остроградського"
author: "&copy; Роменський В'ячеслав"
date: today
format:
  html:
    toc: true
    toc-depth: 4
    toc-location: left
    code-fold: false
    code-tools: true
    embed-resources: true
    smooth-scroll: true
jupyter: python3
---


## Мета роботи

Опанувати базову матричну постановку задачі вибору та навчитися застосовувати класичні критерії прийняття рішень в умовах невизначеності: *Вальда*, *оптимізму (maximax)*, *Байєса–Лапласа* і *Севіджа*.


## Що Ви будете вміти після виконання роботи?

Після виконання лабораторної роботи Ви будете вміти:

- формалізувати задачу вибору у вигляді *матриці виграшів*;
- розрізняти ситуації *повної невизначеності* та *ризику*;
- обчислювати рішення за критеріями *Вальда*, *оптимізму (maximax)*, *Байєса–Лапласа* і *Севіджа*;
- інтерпретувати різницю між песимістичною, оптимістичною та компромісною позиціями особи, що приймає рішення;
- пояснювати, чому різні критерії можуть приводити до різних альтернатив;
- реалізовувати відповідні обчислення засобами *Python*.


### 1. Індивідуальне завдання

Нижче наведено заготовку для *власної матриці виграшів*.  
Ви можете:

- залишити таку саму розмірність, як у базовому прикладі;
- змінити назви альтернатив і станів;
- використати власні значення виграшів;
- за потреби задати свій розподіл імовірностей.

Рекомендується пов'язати матрицю з реальною задачею моніторингу або керування, наприклад:

- вибір стратегії реагування на інцидент;
- вибір режиму моніторингу;
- вибір конфігурації аналітичної системи;
- вибір дії адміністратора в умовах нестабільного інформаційного фону.


In [5]:
# Базові імпорти
import numpy as np
import pandas as pd

student_alternatives = [
    "a1: Пасивний моніторинг",
    "a2: Посилений аналіз журналів",
    "a3: Часткове блокування підозрілих вузлів",
    "a4: Повна ізоляція сегмента мережі",
    "a5: Перехід на резервну інфраструктуру"
]

student_states = [
    "s1: Нормальна робота",
    "s2: Хибна тривога",
    "s3: Локальна атака",
    "s4: Масована атака",
    "s5: Критичний збій систем"
]

student_matrix = np.array([
    [12,  8,   2, -10, -20],
    [10,  9,   8,   0, -12],
    [ 6,  5,  12,   8,  -4],
    [ 0, -2,  10,  15,   6],
    [-6, -8,   6,  14,  20]
], dtype=float)

student_probabilities = np.array([0.60, 0.10, 0.15, 0.10, 0.05], dtype=float)

student_df = pd.DataFrame(student_matrix, index=student_alternatives, columns=student_states)
student_df


,s1: Нормальна робота,s2: Хибна тривога,s3: Локальна атака,s4: Масована атака,s5: Критичний збій систем
a1: Пасивний моніторинг,12.0,8.0,2.0,-10.0,-20.0
a2: Посилений аналіз журналів,10.0,9.0,8.0,0.0,-12.0
a3: Часткове блокування підозрілих вузлів,6.0,5.0,12.0,8.0,-4.0
a4: Повна ізоляція сегмента мережі,0.0,-2.0,10.0,15.0,6.0
a5: Перехід на резервну інфраструктуру,-6.0,-8.0,6.0,14.0,20.0


### 2. Критерій Вальда

Песимістична логіка критерію Вальда полягає в тому, що для кожної альтернативи розглядається її *найгірший можливий результат*.  
Після цього обирається альтернатива, у якої цей найгірший результат є *найкращим серед усіх*.

In [15]:
def student_wald_criterion(matrix, alternatives):
    mins = matrix.min(axis=1) # Minimum for every row
    best = mins.argmax() # Maximum from them
    return pd.DataFrame({
        "Альтернатива": alternatives,
        "Найгірший випадок": mins
    }), alternatives[best], float(mins[best])

student_wald, student_wald_alt, student_wald_val = student_wald_criterion(student_matrix, student_alternatives)

print("Worst possible cases:")
display(student_wald)
print("Best alternative:")
print(student_wald_alt, f"({student_wald_val})")

Worst possible cases:


,Альтернатива,Найгірший випадок
0,a1: Пасивний моніторинг,-20.0
1,a2: Посилений аналіз журналів,-12.0
2,a3: Часткове блокування підозрілих вузлів,-4.0
3,a4: Повна ізоляція сегмента мережі,-2.0
4,a5: Перехід на резервну інфраструктуру,-8.0


Best alternative:
a4: Повна ізоляція сегмента мережі (-2.0)


### 3. Критерій оптимізму (maximax)

Критерій оптимізму орієнтується на *найкращий можливий результат* для кожної альтернативи.  
Такий підхід відповідає ситуації, коли ОПР готова приймати високий ризик заради потенційно найбільшого виграшу.


In [17]:
def student_maximax_criterion(matrix, alternatives):
    maxs = matrix.max(axis=1) # Maximum for every row
    best = maxs.argmax() # Maximum from them
    return pd.DataFrame({
        "Альтернатива": alternatives,
        "Найкращий випадок": maxs
    }), alternatives[best], float(maxs[best])

student_maximax, student_maximax_alt, student_maximax_val = student_maximax_criterion(student_matrix, student_alternatives)

print("Best possible cases:")
display(student_maximax)
print("Best alternative:")
print(student_maximax_alt, f"({student_maximax_val})")

Best possible cases:


,Альтернатива,Найкращий випадок
0,a1: Пасивний моніторинг,12.0
1,a2: Посилений аналіз журналів,10.0
2,a3: Часткове блокування підозрілих вузлів,12.0
3,a4: Повна ізоляція сегмента мережі,15.0
4,a5: Перехід на резервну інфраструктуру,20.0


Best alternative:
a5: Перехід на резервну інфраструктуру (20.0)


### 4. Критерій Байєса–Лапласа

Якщо для станів середовища задано або оцінено ймовірності, ми можемо перейти до обчислення *очікуваного виграшу*.  
Саме це і робить критерій Байєса–Лапласа.

In [37]:
def student_bayes_laplace_criterion(matrix, alternatives, probs):
    # Matrix multiplication
    # Multiplies every row from matrix (by alternatives) with probabilities and sums it up
    expected_values = matrix @ probs
    best_index = expected_values.argmax() # Max expected
    return pd.DataFrame({
        "Альтернатива": alternatives,
        "Математичне сподівання": expected_values
    }), alternatives[best_index], expected_values[best_index]

student_bayes, student_bayes_alt, student_bayes_val = student_bayes_laplace_criterion(student_matrix, student_alternatives, student_probabilities)

print("Average by alternatives:")
display(student_bayes)
print("Best alternative by mean:")
print(student_bayes_alt, f"({student_bayes_val})")

Average by alternatives:


,Альтернатива,Математичне сподівання
0,a1: Пасивний моніторинг,6.3
1,a2: Посилений аналіз журналів,7.5
2,a3: Часткове блокування підозрілих вузлів,6.5
3,a4: Повна ізоляція сегмента мережі,3.1
4,a5: Перехід на резервну інфраструктуру,-1.1


Best alternative by mean:
a2: Посилений аналіз журналів (7.5)


### 5. Критерій Севіджа

На відміну від попередніх критеріїв, критерій Севіджа працює з *жалем*, тобто з втратами від того, що ми не обрали найкращу альтернативу для конкретного стану середовища.


In [41]:
def student_savage_criterion(matrix, alternatives, states):
    column_maxs = matrix.max(axis=0)
    regret_matrix = column_maxs - matrix # differences between every value and maxvalue in column
    row_regret_maxs = regret_matrix.max(axis=1) # max regret for every row
    best_index = row_regret_maxs.argmin() # looks for minimum regret

    regret_df = pd.DataFrame(regret_matrix, index=alternatives, columns=states)
    summary_df = pd.DataFrame({
        "Альтернатива": alternatives,
        "Максимальний жаль": row_regret_maxs
    })
    return regret_df, summary_df, alternatives[best_index], row_regret_maxs[best_index]

student_regret, student_savage, student_savage_alt, student_savage_val = student_savage_criterion(student_matrix, student_alternatives, student_states)

print("Regret matrix:")
display(student_regret)
print("Max regret for every alternative:")
display(student_savage)
print("Best alternative by savage:")
print(student_savage_alt, f"({student_savage_val})")

Regret matrix:


,s1: Нормальна робота,s2: Хибна тривога,s3: Локальна атака,s4: Масована атака,s5: Критичний збій систем
a1: Пасивний моніторинг,0.0,1.0,10.0,25.0,40.0
a2: Посилений аналіз журналів,2.0,0.0,4.0,15.0,32.0
a3: Часткове блокування підозрілих вузлів,6.0,4.0,0.0,7.0,24.0
a4: Повна ізоляція сегмента мережі,12.0,11.0,2.0,0.0,14.0
a5: Перехід на резервну інфраструктуру,18.0,17.0,6.0,1.0,0.0


Max regret for every alternative:


,Альтернатива,Максимальний жаль
0,a1: Пасивний моніторинг,40.0
1,a2: Посилений аналіз журналів,32.0
2,a3: Часткове блокування підозрілих вузлів,24.0
3,a4: Повна ізоляція сегмента мережі,14.0
4,a5: Перехід на резервну інфраструктуру,18.0


Best alternative by savage:
a4: Повна ізоляція сегмента мережі (14.0)


### 6. Порівняння усіх критеріїв

Показово, що майже кожний варіант представив різні значення, різну оцінку та відповідь.


In [42]:
student_summary = pd.DataFrame({
    "Критерій": ["Вальда", "Оптимізму (maximax)", "Байєса–Лапласа", "Севіджа"],
    "Оптимальна альтернатива": [student_wald_alt, student_maximax_alt, student_bayes_alt, student_savage_alt],
    "Оцінка": [
        student_wald_val,
        student_maximax_val,
        round(student_bayes_val, 3),
        student_savage_val
    ]
})

student_summary


,Критерій,Оптимальна альтернатива,Оцінка
0,Вальда,a4: Повна ізоляція сегмента мережі,-2.0
1,Оптимізму (maximax),a5: Перехід на резервну інфраструктуру,20.0
2,Байєса–Лапласа,a2: Посилений аналіз журналів,7.5
3,Севіджа,a4: Повна ізоляція сегмента мережі,14.0


## 10. Висновки до роботи

Матриця виграшів була обрана для ситуації, коли треба реагувати на кібератаки, пов'язані із нападом на інформаційну систему. Альтернативи варіюються від стабільного спостереження до повного переходу на резервну інфраструктуру. Оцінки були обрані згідно того, наскільки добре чи погано альтернатива підходить для реагування на отриману ситуацію. Також було додано масив, що відображає ймовірності появи цих станів. Стани не є рівномірно розподіленими між собою: перший стан має 60% появи, коли останній 5%.

За критерієм Вальда (песимістичним) ми знаходимо найгірший можливий випадок і найкращу реакцію на нього. Виявляється, що вибір а4 дає найбільший виграш, якщо наша удача є найгіршою. Оцінка такого вибору дорівнює -2.
За критерієм оптимізму ми знаходимо найкращий випадок і найкращу реакцію на нього. Очевидно, що якщо обрати найризикованіший варіант, то виграш буде найбільшим. Оцінка такого вибору дорівнює 20.
За критерієм Байєса-Лапласа ми нарешті беремо до уваги те, що кожен стан з'являється не однаково часто. Цей критерій дозволяє знайти найбільш стабільний середній виграш, яким є альтернатива а4 та оцінка 7.5.
За критерієм Севіджа ми беремо до уваги те, що ми щось втрачаємо, коли не обираємо найкращу альтернативу при її наявності. Альтернатива із мінімальним жалем - це а4 із оцінкою 14, що також є відповіддю песимістичного критерію.

Якщо зробити висновки, то треба зазначити що:
- критерій оптимізму розраховує на найкращий результат, отже підходить для ризикованих дій із потенційно найбільшим виграшем; його можна обирати, коли треба короткостроковий виграш;
- критерій песимізму дає варіант не отримати гіршу оцінку, ніж -2, отже більш передбачуваний та дуже безпечний; його можна обирати, коли треба уникнути будь-яких сильних програшів;
- критерій Байєса-Лапласа є більш реалістичним для реального світу, де стани мають різну ймовірність появи; його можна обирати, коли важливий довгостроковий виграш;
- критерій Севіджа мінімізує 'жаль' чи втрати від того, що було обрано не найкращий варіант для даної ситуації; його можна обирати, коли треба не тільки не програвати, а і ще не сильно відставати від конкурентів, які могли обрати кращий варіант за вас; можна помітити, що відповідь за критерієм Севіджа така ж, як і за критерієм песимізму, бо також враховує найбезпечніший випадок;

Отже, для даної ситуації найкращим є варіант критерію Байєса-Лапласа.


## Контрольні питання

1. Що означає подати задачу вибору у вигляді *матриці виграшів*?  
Матриця виграшів — це таблиця, де рядки відповідають вашим можливим стратегіям, а стовпці — станам середовища, на які ви не маєте впливу. На перетині цих значень вказується кількісний результат (прибуток або користь), який ви отримаєте за певної комбінації дій та обставин.

2. Яку позицію щодо невизначеності реалізує критерій *Вальда*?  
Критерій Вальда реалізує позицію крайнього песимізму, орієнтуючись на найгірший можливий розвиток подій для кожної альтернативи. Вибір здійснюється за принципом «максимін», тобто ви обираєте той варіант, де мінімальний виграш є найбільшим серед усіх можливих невдач.

3. Чим критерій *оптимізму (maximax)* відрізняється від критерію Вальда?  
Критерій максимаксу є стратегією абсолютного оптимізму, яка ігнорує ризики та орієнтується лише на максимально можливий успіх. На відміну від Вальда, який шукає «найкраще серед найгіршого», цей підхід обирає «найкраще серед найкращого».

4. У чому полягає зміст критерію *Байєса–Лапласа*?  
Критерій Байєса–Лапласа базується на припущенні, що всі стани середовища є рівноймовірними або мають відомий розподіл імовірностей. Оптимальним рішенням вважається те, яке забезпечує найбільше середнє значення (математичне очікування) виграшу.

5. Що таке *матриця жалю* в критерії Севіджа?  
Матриця жалю відображає не прямі доходи, а величину втраченої вигоди, тобто різницю між максимально можливим виграшем у конкретній ситуації та результатом обраної дії. Вона дозволяє оцінити, наскільки ви «пошкодуєте» про своє рішення, якщо настане певний стан середовища.

6. Чому різні критерії можуть вказувати на різні альтернативи?  
Різні критерії відображають різну схильність особи до ризику та різні психологічні установки (від обережності до азарту). Оскільки кожен математичний підхід по-своєму оцінює вагомість загроз та можливостей, один і той самий набір даних може призводити до різних «найкращих» рішень.

7. У яких прикладних задачах моніторингу такі критерії є корисними?
У задачах моніторингу ці критерії корисні для планування систем захисту об'єктів, прогнозування екологічних ризиків або вибору стратегій реагування на надзвичайні ситуації. Вони допомагають визначити оптимальну кількість датчиків або періодичність перевірок в умовах, коли поведінка об'єкта чи природи є непередбачуваною.

# Список літератури

1. Шевченко І. В. Методичні вказівки щодо виконання лабораторних робіт з освітнього компонента «Моніторинг та керування в слабкоструктурованих процесах і системах» для здобувачів вищої освіти денної форми навчання зі спеціальності F3 – «Комп’ютерні науки» освітньо-професійної програми «Інформаційні управляючі системи та технології» другого (магістерського) освітнього рівня, 2025.
2. Бідюк П. І., Тимощук О. Л., Коваленко А. Є., Коршевнюк Л. О. *Системи і методи підтримки прийняття рішень*. Київ: КПІ ім. Ігоря Сікорського, 2022. 
3. Творошенко І. С. *Технології прийняття рішень в інформаційних системах*. Харків: ХНУРЕ, 2021.  
4. Гнатенко В. М., Снитюк В. Є. *Експертні технології прийняття рішень*. Київ: ТОВ «Маклаут».  
5. Конспект лекції 5 з освітнього компонента *«Моніторинг та керування в слабкоструктурованих процесах і системах»*.
